# Binance trades and aggressor direction

This example replays raw Binance trades through `fiml`. Binance's `market_maker` field means **the buyer is the maker**. `fiml` transforms it into `trade_direction`: `+1` for a buyer-initiated trade and `-1` for a seller-initiated trade. An EMA of that value is a bounded order-flow signal: positive values indicate more aggressive buying and negative values indicate more aggressive selling.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import fiml

In [ ]:
working_dir = Path.cwd()
notebook_dir = (
    working_dir
    if (working_dir / "price_data_binance_trades.csv").exists()
    else working_dir / "notebooks"
)
trades_path = notebook_dir / "price_data_binance_trades.csv"
assert trades_path.exists(), f"Could not find {trades_path}"

In [ ]:
raw_trades = pd.read_csv(trades_path, parse_dates=["trade_time"])
trades = (
    raw_trades
    # The CSV is newest-first. Replay oldest-first, preserving trade order
    # for rows with the same millisecond timestamp.
    .sort_values(["trade_time", "trade_id"], kind="stable")
    .rename(columns={"quantity": "volume"})
    .assign(ts=lambda frame: frame["trade_time"].astype("int64") // 1_000_000)
    [["symbol", "ts", "price", "volume", "market_maker"]]
    .reset_index(drop=True)
)

assert trades["ts"].is_monotonic_increasing
assert pd.api.types.is_bool_dtype(trades["market_maker"])
trades.head()

In [ ]:
symbol = trades["symbol"].iat[0]
feature_set = (
    fiml.FeatureSet()
    .ema(symbol, [8], source="trade_direction")
    .ema(symbol, [8], source="trade_price")
    .trade_count_timed(symbol, aggregation="1ms", window="10s")
)
extractor = fiml.FeatureExtractor(feature_set, output_dtype=np.float32)
features = extractor.compute_features(trades, market_maker="market_maker")
features.head()

In [ ]:
direction_feature = f"{symbol}:trade_direction:ema:8"
preview = pd.concat(
    [trades[["price", "volume", "market_maker"]], features[[direction_feature]]],
    axis=1,
)

assert features.index.equals(trades.index)
assert features[direction_feature].between(-1.0, 1.0).all()
preview